# 5G hypothesis assessment — testing both directions

The question on the table: *can we say from these data that bumblebees are
hindered by 5G radiation?*

This notebook deliberately runs the analysis from **both** sides:

- **Supporting hypothesis (H1):** ON exposure depresses foraging activity /
  return ratio / movement quality on the hive closest to the antenna (939),
  with a dose-response on actual dBm.
- **Null / alternative explanations (H0):** the apparent effect on 939 is
  driven by **temperature**, **seasonal/date trend**, **non-independence**
  of 3-day cycle blocks, **multiple-testing**, or **chance with small n**.

The structure mirrors how a reviewer would read this:

1. **Treatment verification.** Did ON days actually have measurably higher
   dBm than OFF days? If not, the "treatment" is fictional.
2. **Confounder check 1 — temperature.** Bumblebees are highly
   temperature-sensitive. If ON and OFF days had different temperatures,
   that alone could explain everything.
3. **Confounder check 2 — date trend.** Late April vs early May has
   different flower availability, weather, and bee maturity.
4. **Support evidence.** Effect-size + significance per indicator, system,
   with and without BH correction. Dose-response with continuous dBm.
5. **Pressure-test evidence.** Cycle-block permutation test (correct unit
   of analysis), regression with temperature & date as covariates,
   cross-system inconsistency.
6. **BASELINE robustness.** Is the BASELINE reference meaningful, or is it
   dominated by a single high-activity day (2026-04-18)?
7. **Verdict table.** Per claim: supporting evidence, opposing evidence,
   confidence level.

The verdict cell at the end should let your professor see exactly what can
and cannot be claimed.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# Paths
# ---------------------------------------------------------------------------

V3_DIR    = Path("data/multi_day_v3")
DATA_ROOT = Path("../../data")

CYCLE_ANCHOR   = pd.Timestamp("2026-04-23")
CYCLE_ON_DAYS  = 3
CYCLE_OFF_DAYS = 3
CYCLE_LEN      = CYCLE_ON_DAYS + CYCLE_OFF_DAYS  # 6
SYSTEMS = [900, 939]

# Bumblebees foraged from sensor 6 = system 900, sensor 4 = system 939
SENSOR_TO_SYSTEM = {6: 900, 4: 939}

N_PERMUTATIONS = 5000
RNG = np.random.default_rng(20260515)


def condition_for(date_like):
    d = pd.Timestamp(date_like)
    if d < CYCLE_ANCHOR:
        return "BASELINE"
    days_since = (d - CYCLE_ANCHOR).days
    return "ON" if (days_since % CYCLE_LEN) < CYCLE_ON_DAYS else "OFF"


def cycle_block_for(date_like):
    """Integer ID of the 3-day block this date belongs to. BASELINE = -k
    for ordered baseline triplets; ON/OFF blocks indexed 0,1,2,..."""
    d = pd.Timestamp(date_like)
    if d < CYCLE_ANCHOR:
        days_before = (CYCLE_ANCHOR - d).days
        return -((days_before - 1) // CYCLE_ON_DAYS + 1)
    days_since = (d - CYCLE_ANCHOR).days
    return days_since // CYCLE_ON_DAYS    # 3-day blocks, alternating ON/OFF


print(f"V3 dir:    {V3_DIR.resolve()}  exists={V3_DIR.exists()}")
print(f"Data root: {DATA_ROOT.resolve()}  exists={DATA_ROOT.exists()}")


In [ ]:

# Load: v3 daily + tracks, dBm, temperature
# ---------------------------------------------------------------------------

daily  = pd.read_csv(V3_DIR / "daily_summary.csv")
tracks = pd.read_csv(V3_DIR / "per_track_indicators.csv", parse_dates=["ts"])

daily["condition"]   = daily["date"].apply(condition_for)
tracks["condition"]  = tracks["date"].apply(condition_for)
daily["cycle_block"] = daily["date"].apply(cycle_block_for)
daily["date_ts"]     = pd.to_datetime(daily["date"])
daily["date_index"]  = (daily["date_ts"] - daily["date_ts"].min()).dt.days
daily["exit_return_count"] = daily["n_exit_v3"] + daily["n_return_v3"]


# --- dBm per (date, system) ----
dbm_files = sorted(DATA_ROOT.glob("Power levels (dBm)*.csv"))
if dbm_files:
    raw_dbm = pd.read_csv(dbm_files[-1])
    raw_dbm["timestamp"] = pd.to_datetime(raw_dbm["timestamp"])
    raw_dbm["date"]      = raw_dbm["timestamp"].dt.strftime("%Y-%m-%d")
    raw_dbm["system_id"] = raw_dbm["sensor"].map(SENSOR_TO_SYSTEM)
    dbm_daily = (raw_dbm.dropna(subset=["system_id"])
                        .groupby(["date", "system_id"])
                        .agg(mean_dbm=("dBm","mean"),
                             max_dbm =("dBm","max"),
                             p95_dbm=("dBm", lambda s: s.quantile(0.95)))
                        .reset_index())
    dbm_daily["system_id"] = dbm_daily["system_id"].astype(int)
    daily = daily.merge(dbm_daily, on=["date", "system_id"], how="left")
    print(f"dBm: loaded {len(raw_dbm):,} rows, joined per (date, system)")
else:
    print("(no dBm file found — dose-response cells will be skipped)")


# --- KNMI temperature (hourly) → daytime mean per date ----
WIND_FILE = DATA_ROOT / "wind_data_04-19_to_05-11.txt"
if WIND_FILE.exists():
    # The file has a column header line that starts with "# STN,YYYYMMDD,HH,..."
    # Real data lines start with the station code "330,". Read them directly.
    cols = ["STN","YYYYMMDD","HH","DD","FH","FF","FX","T","T10N","TD","SQ","Q",
            "DR","RH","P","VV","N","U","WW","IX","M","R","S","O","Y"]
    wx = pd.read_csv(WIND_FILE, comment="#", header=None, names=cols,
                     skipinitialspace=True, na_values=[""])
    wx["T_C"]  = pd.to_numeric(wx["T"], errors="coerce") / 10.0
    wx["date"] = pd.to_datetime(wx["YYYYMMDD"].astype(int).astype(str),
                                 format="%Y%m%d").dt.strftime("%Y-%m-%d")
    wx["HH"]   = pd.to_numeric(wx["HH"], errors="coerce")
    # daytime mean = 08:00 → 18:00 (foraging window)
    daytime = wx[(wx["HH"] >= 8) & (wx["HH"] <= 18)]
    temp_daily = (daytime.groupby("date")
                          .agg(temp_mean=("T_C","mean"),
                               temp_max=("T_C","max"),
                               temp_min=("T_C","min"))
                          .reset_index())
    daily = daily.merge(temp_daily, on="date", how="left")
    print(f"Temperature: {len(temp_daily)} days with daytime data")
else:
    print(f"(no temperature file found — confounder check will be skipped)")
    daily["temp_mean"] = np.nan


print()
print(f"daily: {daily.shape}")
print(daily[["date","system_id","condition","cycle_block",
             "n_exit_v3","n_return_v3","re_ratio_v3","mean_dbm","temp_mean"]]
      .head(8).to_string(index=False))


## Section 1 — Treatment verification (dBm)

In [ ]:

# Section 1 — Treatment verification: did ON ≠ OFF in dBm?
# ---------------------------------------------------------------------------
# If the manipulation was real, ON days must measurably exceed OFF days in
# radiation level. If they don\'t, the entire claim falls before it starts.

if "mean_dbm" not in daily.columns or daily["mean_dbm"].isna().all():
    print("No dBm data — skipping treatment verification")
else:
    fig, axes = plt.subplots(1, len(SYSTEMS), figsize=(5 * len(SYSTEMS), 4),
                             squeeze=False)
    rows = []
    for c, sys_id in enumerate(SYSTEMS):
        ax = axes[0, c]
        sub = daily[daily["system_id"] == sys_id]
        groups = {k: sub.loc[sub["condition"] == k, "mean_dbm"].dropna().values
                  for k in ["BASELINE", "OFF", "ON"]}
        bp = ax.boxplot([groups[k] for k in ["BASELINE", "OFF", "ON"]],
                         tick_labels=["BASELINE", "OFF", "ON"],
                         patch_artist=True, widths=0.55, showmeans=True)
        for patch, k in zip(bp["boxes"], ["BASELINE", "OFF", "ON"]):
            patch.set_facecolor({"BASELINE": "#888", "OFF": "tab:green",
                                  "ON": "tab:red"}[k])
            patch.set_alpha(0.55)
        ax.set_title(f"system {sys_id}  (sensor {[k for k,v in SENSOR_TO_SYSTEM.items() if v==sys_id][0]})")
        ax.set_ylabel("daily mean dBm")
        ax.grid(axis="y", alpha=0.3)
        if len(groups["ON"]) > 1 and len(groups["OFF"]) > 1:
            u, p = stats.mannwhitneyu(groups["ON"], groups["OFF"],
                                       alternative="two-sided")
            delta = float(np.median(groups["ON"]) - np.median(groups["OFF"]))
            rows.append((sys_id, len(groups["OFF"]), len(groups["ON"]),
                         float(np.median(groups["OFF"])),
                         float(np.median(groups["ON"])),
                         delta, float(p)))
            ax.text(0.02, 0.95, f"ON−OFF Δmed = {delta:+.1f} dBm\np(MWU) = {p:.4f}",
                    transform=ax.transAxes, va="top", fontsize=9,
                    bbox=dict(facecolor="white", edgecolor="gray", alpha=0.85))

    plt.suptitle("Treatment verification — daily mean dBm by condition",
                 fontsize=12, y=1.0)
    plt.tight_layout()
    plt.show()

    print()
    print("Treatment verification table:")
    print(pd.DataFrame(rows, columns=["system","n_OFF","n_ON","med_OFF","med_ON",
                                       "delta","p_MWU"]).round(3).to_string(index=False))


## Section 2 — Confounder: temperature

In [ ]:

# Section 2 — Confounder check: temperature × condition
# ---------------------------------------------------------------------------
# Bumblebees forage less when it\'s cold. If ON days happened to be cooler
# than OFF days, "5G impairs foraging" and "cold impairs foraging" become
# indistinguishable.

if "temp_mean" not in daily.columns or daily["temp_mean"].isna().all():
    print("No temperature data — skipping")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Boxplot temperature by condition (one row per condition,
    # collapsed across systems since temp is the same in both)
    temp_table = (daily[["date","temp_mean","condition"]]
                  .drop_duplicates("date"))
    groups = {k: temp_table.loc[temp_table["condition"] == k, "temp_mean"].dropna()
              for k in ["BASELINE", "OFF", "ON"]}
    bp = axes[0].boxplot([groups[k].values for k in ["BASELINE", "OFF", "ON"]],
                          tick_labels=["BASELINE", "OFF", "ON"], patch_artist=True,
                          widths=0.55, showmeans=True)
    for patch, k in zip(bp["boxes"], ["BASELINE", "OFF", "ON"]):
        patch.set_facecolor({"BASELINE": "#888", "OFF": "tab:green",
                              "ON": "tab:red"}[k])
        patch.set_alpha(0.55)
    axes[0].set_title("Daytime mean temperature by condition")
    axes[0].set_ylabel("T (°C)")
    axes[0].grid(axis="y", alpha=0.3)
    for i, k in enumerate(["BASELINE", "OFF", "ON"], start=1):
        axes[0].text(i, axes[0].get_ylim()[0],
                      f"n={len(groups[k])}", ha="center", va="bottom",
                      fontsize=8, color="#444")

    # Scatter: temperature vs exit_return_count per (date, system)
    for sys_id, marker in [(900, "o"), (939, "s")]:
        sub = daily[daily["system_id"] == sys_id]
        axes[1].scatter(sub["temp_mean"], sub["exit_return_count"],
                         label=f"system {sys_id}", alpha=0.7, marker=marker, s=40)
    axes[1].set_xlabel("daytime mean T (°C)")
    axes[1].set_ylabel("v3 exits + returns / day")
    axes[1].set_title("Activity vs temperature (pooled across conditions)")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    # Stat tests
    on  = groups["ON"]
    off = groups["OFF"]
    if len(on) > 1 and len(off) > 1:
        u, p = stats.mannwhitneyu(on, off, alternative="two-sided")
        print(f"Temperature ON vs OFF: median(ON) = {on.median():.1f} °C,  "
              f"median(OFF) = {off.median():.1f} °C,  p(MWU) = {p:.4f}")

    # Spearman correlation between temp and activity per system
    print()
    print("Spearman ρ (temperature ↔ exit_return_count) — pooled across conditions:")
    for sys_id in SYSTEMS:
        sub = daily[(daily["system_id"] == sys_id)
                    & daily["temp_mean"].notna()
                    & daily["exit_return_count"].notna()]
        if len(sub) > 5:
            rho, p = stats.spearmanr(sub["temp_mean"], sub["exit_return_count"])
            print(f"  system {sys_id}: ρ = {rho:+.3f}  p = {p:.4f}  (n = {len(sub)})")


## Section 3 — Confounder: date / seasonal trend

In [ ]:

# Section 3 — Confounder check: date / seasonal trend
# ---------------------------------------------------------------------------
# Is activity changing systematically over time independent of condition?
# If ON tends to fall later in the experiment (or earlier), a seasonal
# trend will look identical to a 5G effect.

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for sys_id, marker in [(900, "o"), (939, "s")]:
    sub = daily[daily["system_id"] == sys_id].sort_values("date_ts")
    colors = sub["condition"].map({"BASELINE": "#888", "OFF": "tab:green",
                                     "ON": "tab:red"})
    axes[0].scatter(sub["date_ts"], sub["exit_return_count"], c=colors,
                     marker=marker, s=42, label=f"sys {sys_id}",
                     edgecolor="black", linewidth=0.4)
    axes[1].scatter(sub["date_ts"], sub["re_ratio_v3"], c=colors,
                     marker=marker, s=42, edgecolor="black", linewidth=0.4)
axes[0].axvline(CYCLE_ANCHOR, linestyle="--", color="black", alpha=0.4,
                 label="cycle anchor")
axes[1].axvline(CYCLE_ANCHOR, linestyle="--", color="black", alpha=0.4)
axes[0].set_ylabel("exit + return / day")
axes[1].set_ylabel("return / exit ratio (v3)")
axes[1].set_xlabel("date")
axes[0].set_title("Activity over time — colored by condition (grey=BASELINE, green=OFF, red=ON)")
axes[0].legend(loc="upper right", fontsize=8)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Test for date trend in indicator, pooled across conditions
print("Spearman ρ (date_index ↔ indicator) — pooled (after CYCLE_ANCHOR):")
for sys_id in SYSTEMS:
    sub = daily[(daily["system_id"] == sys_id)
                & (daily["condition"].isin(["ON", "OFF"]))]
    if len(sub) > 5:
        rho1, p1 = stats.spearmanr(sub["date_index"], sub["exit_return_count"])
        rho2, p2 = stats.spearmanr(sub["date_index"], sub["re_ratio_v3"])
        print(f"  system {sys_id}: count vs date ρ={rho1:+.3f} p={p1:.3f}    "
              f"re_ratio vs date ρ={rho2:+.3f} p={p2:.3f}    (n={len(sub)})")


## Section 4 — SUPPORT: per-system effect sizes

In [ ]:

# Section 4 — SUPPORT: per-system effect sizes (uncorrected + BH-corrected)
# ---------------------------------------------------------------------------

from scipy.stats import false_discovery_control

def rank_biserial(on_vals, off_vals):
    on_v, off_v = np.asarray(on_vals), np.asarray(off_vals)
    if len(on_v) == 0 or len(off_v) == 0:
        return np.nan
    u, _ = stats.mannwhitneyu(on_v, off_v, alternative="two-sided")
    return 1.0 - (2.0 * u) / (len(on_v) * len(off_v))


# Compute per-day tortuosity & IFI-CV from tracks
ifi = (tracks.groupby(["date", "system_id"])
              .apply(lambda g: (pd.to_datetime(
                  g.loc[g["hive_exit_v3"], "ts"]).sort_values().diff()
                                .dropna().dt.total_seconds().std()
                                / pd.to_datetime(
                  g.loc[g["hive_exit_v3"], "ts"]).sort_values().diff()
                                .dropna().dt.total_seconds().mean())
                                if g["hive_exit_v3"].sum() >= 3 else np.nan,
                       include_groups=False)
              .reset_index(name="ifi_cv"))
tort = (tracks.groupby(["date", "system_id"])["tortuosity"]
              .median().reset_index(name="path_tortuosity"))

contrast = (daily.merge(ifi,  on=["date","system_id"], how="left")
                  .merge(tort, on=["date","system_id"], how="left"))
contrast["re_ratio"] = contrast["re_ratio_v3"]

INDICATORS = [
    ("exit_return_count", "exit + return / day"),
    ("re_ratio",          "return / exit ratio"),
    ("path_tortuosity",   "median path tortuosity"),
    ("ifi_cv",            "IFI coefficient of variation"),
]

rows = []
for sys_id in SYSTEMS:
    sub = contrast[(contrast["system_id"] == sys_id)
                   & contrast["condition"].isin(["ON", "OFF"])]
    p_list = []
    rec = []
    for col, label in INDICATORS:
        on  = sub.loc[sub["condition"] == "ON",  col].dropna()
        off = sub.loc[sub["condition"] == "OFF", col].dropna()
        if len(on) < 2 or len(off) < 2:
            p_list.append(np.nan)
            rec.append((sys_id, label, np.nan, np.nan, np.nan, np.nan,
                         len(on), len(off), float(on.mean() if len(on) else np.nan),
                         float(off.mean() if len(off) else np.nan)))
            continue
        u, p = stats.mannwhitneyu(on, off, alternative="two-sided")
        r = rank_biserial(on, off)
        p_list.append(p)
        rec.append((sys_id, label, float(u), float(p), float(r),
                     "ON>OFF" if on.mean() > off.mean() else "OFF>ON",
                     len(on), len(off), float(on.mean()), float(off.mean())))
    # BH per system across the 4 indicators
    valid = [p for p in p_list if not pd.isna(p)]
    bh    = false_discovery_control(valid) if valid else []
    bh_iter = iter(bh)
    bh_all  = [float(next(bh_iter)) if not pd.isna(p) else np.nan for p in p_list]
    for r, p_bh in zip(rec, bh_all):
        rows.append((*r, p_bh,
                      "directional" if (abs(r[4]) > 0.3) else "—"))

effect_table = pd.DataFrame(rows, columns=[
    "system","indicator","U","p","r","direction","n_ON","n_OFF",
    "mean_ON","mean_OFF","p_BH","flag"])
print("Per-system effect sizes (positive r = ON > OFF):")
print(effect_table.round(3).to_string(index=False))

# Bar chart of |r| per indicator per system
fig, ax = plt.subplots(1, 1, figsize=(9, 4))
plot_df = effect_table.dropna(subset=["r"])
x = np.arange(len(INDICATORS))
w = 0.38
for i, sys_id in enumerate(SYSTEMS):
    sub = plot_df[plot_df["system"] == sys_id]
    ind_order = [lbl for _, lbl in INDICATORS]
    rs = [sub.loc[sub["indicator"] == lbl, "r"].values[0]
          if (sub["indicator"] == lbl).any() else 0 for lbl in ind_order]
    ax.bar(x + (-w/2 + i*w), rs, width=w, label=f"system {sys_id}",
            color=["#1f77b4","#d62728"][i], alpha=0.8)
ax.axhline(0, color="black", linewidth=0.6)
ax.axhline( 0.3, linestyle=":", color="black", alpha=0.4, label="|r|=0.3 (directional)")
ax.axhline(-0.3, linestyle=":", color="black", alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels([lbl for _, lbl in INDICATORS], rotation=20, ha="right")
ax.set_ylabel("rank-biserial r  (ON vs OFF, +=ON higher)")
ax.set_title("Effect sizes per indicator per system")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Section 5 — SUPPORT: dose-response (continuous dBm)

In [ ]:

# Section 5 — SUPPORT: dose-response (continuous dBm vs indicators)
# ---------------------------------------------------------------------------
# If the effect is real, it should track the actual radiation level, not
# just the ON/OFF schedule. Spearman ρ between daily mean dBm (per system)
# and each indicator. Negative ρ = more dBm → less activity = consistent
# with the H1 direction.

if "mean_dbm" not in contrast.columns or contrast["mean_dbm"].isna().all():
    print("No dBm data — skipping dose-response")
else:
    fig, axes = plt.subplots(len(INDICATORS), len(SYSTEMS),
                              figsize=(5 * len(SYSTEMS), 3.0 * len(INDICATORS)),
                              squeeze=False)
    rows = []
    for r, (col, label) in enumerate(INDICATORS):
        for c, sys_id in enumerate(SYSTEMS):
            ax = axes[r, c]
            sub = contrast[(contrast["system_id"] == sys_id)
                            & contrast["mean_dbm"].notna()
                            & contrast[col].notna()]
            if len(sub) < 5:
                ax.set_title(f"sys {sys_id} — {label} (n<5)")
                continue
            colors = sub["condition"].map({"BASELINE": "#888", "OFF": "tab:green",
                                             "ON": "tab:red"})
            ax.scatter(sub["mean_dbm"], sub[col], c=colors, s=42,
                        edgecolor="black", linewidth=0.4)
            try:
                rho, p = stats.spearmanr(sub["mean_dbm"], sub[col])
                rows.append((sys_id, label, float(rho), float(p), len(sub)))
                ax.set_title(f"sys {sys_id} — {label}\nρ = {rho:+.3f}  p = {p:.4f}  (n={len(sub)})")
            except Exception:
                ax.set_title(f"sys {sys_id} — {label}")
            ax.set_xlabel("daily mean dBm")
            ax.set_ylabel(label)
            ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print()
    print("Dose-response (Spearman, pooled across conditions):")
    print(pd.DataFrame(rows, columns=["system","indicator","rho","p","n"])
            .round(3).to_string(index=False))


## Section 6 — DISPROVE: cross-system inconsistency

In [ ]:

# Section 6 — DISPROVE: system 900 vs 939 cross-system inconsistency
# ---------------------------------------------------------------------------
# Both hives are in the same greenhouse, exposed to the same air, same
# flowers, same lights — they differ in their distance to the 5G antenna
# (939 is closer, per the user). If 5G "hinders bumblebees" as a general
# claim, an effect should be visible on BOTH systems with magnitude scaling
# with dBm. If 900 is null and 939 shows a directional effect, that\'s
# *consistent* with a dose-response — but it\'s also consistent with
# "system 939 happens to be noisier".

agree = []
for col, label in INDICATORS:
    e900 = effect_table[(effect_table["system"] == 900)
                         & (effect_table["indicator"] == label)]
    e939 = effect_table[(effect_table["system"] == 939)
                         & (effect_table["indicator"] == label)]
    if e900.empty or e939.empty:
        continue
    r900, p900 = float(e900["r"].values[0]), float(e900["p"].values[0])
    r939, p939 = float(e939["r"].values[0]), float(e939["p"].values[0])
    agree.append({
        "indicator":    label,
        "r_900":         r900,  "p_900": p900,
        "r_939":         r939,  "p_939": p939,
        "directional_900": abs(r900) > 0.3,
        "directional_939": abs(r939) > 0.3,
        "same_direction": (np.sign(r900) == np.sign(r939)) if (not pd.isna(r900) and not pd.isna(r939)) else False,
    })

cross = pd.DataFrame(agree)
print("Cross-system agreement (an effect should show up on BOTH systems with same direction):")
print(cross.round(3).to_string(index=False))

# Visualize: paired r values
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
xs = np.arange(len(cross))
ax.plot([xs, xs], [cross["r_900"], cross["r_939"]], color="gray", alpha=0.4)
ax.scatter(xs, cross["r_900"], s=80, label="system 900", color="#1f77b4",
            zorder=3, edgecolor="black", linewidth=0.4)
ax.scatter(xs, cross["r_939"], s=80, label="system 939", color="#d62728",
            zorder=3, edgecolor="black", linewidth=0.4)
ax.axhline(0, color="black", linewidth=0.6)
ax.axhline( 0.3, linestyle=":", color="black", alpha=0.4)
ax.axhline(-0.3, linestyle=":", color="black", alpha=0.4)
ax.set_xticks(xs)
ax.set_xticklabels(cross["indicator"], rotation=20, ha="right")
ax.set_ylabel("rank-biserial r (ON vs OFF)")
ax.set_title("Paired effect sizes: do 900 and 939 agree on direction?")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Section 7 — DISPROVE: cycle-block permutation test

In [ ]:

# Section 7 — DISPROVE: cycle-block permutation test
# ---------------------------------------------------------------------------
# Days within a 3-day block aren\'t independent (same bees, same weather,
# autocorrelated activity). Day-level shuffling under the null overstates
# significance. The correct unit is the *block*. We shuffle the ON/OFF
# label across blocks, recompute the test statistic, and see how often the
# permuted statistic is at least as extreme as the observed one.

post = contrast[contrast["condition"].isin(["ON", "OFF"])].copy()
post["cycle_block"] = post["date"].apply(cycle_block_for)

# Identify each block and its actual label
block_labels = (post.drop_duplicates("cycle_block")
                    [["cycle_block","condition"]].copy())
print(f"Blocks (post-anchor): {len(block_labels)}")
print(block_labels.to_string(index=False))


def permutation_p(values_by_block_and_system, n_perm=N_PERMUTATIONS):
    """For each system, the test statistic is (median ON − median OFF).
    Shuffle block-labels n_perm times; compute two-sided p."""
    out = {}
    for sys_id, df in values_by_block_and_system.items():
        on_blocks  = df.loc[df["condition"] == "ON",  "cycle_block"].unique()
        off_blocks = df.loc[df["condition"] == "OFF", "cycle_block"].unique()
        all_blocks = np.concatenate([on_blocks, off_blocks])
        n_on_blocks = len(on_blocks)
        observed = (df.loc[df["condition"] == "ON",  "value"].median()
                    - df.loc[df["condition"] == "OFF", "value"].median())
        more_extreme = 0
        for _ in range(n_perm):
            shuffled_on = set(RNG.choice(all_blocks, size=n_on_blocks, replace=False))
            label = df["cycle_block"].isin(shuffled_on).map({True: "ON", False: "OFF"})
            stat = (df.loc[label == "ON",  "value"].median()
                    - df.loc[label == "OFF", "value"].median())
            if abs(stat) >= abs(observed) - 1e-12:
                more_extreme += 1
        out[sys_id] = (float(observed), more_extreme / n_perm)
    return out


perm_rows = []
for col, label in INDICATORS:
    grouped = {}
    for sys_id in SYSTEMS:
        sub = post[(post["system_id"] == sys_id) & post[col].notna()]
        if len(sub) < 6:
            continue
        grouped[sys_id] = sub.assign(value=sub[col])[["cycle_block","condition","value"]]
    if not grouped:
        continue
    res = permutation_p(grouped)
    for sys_id, (obs, p_perm) in res.items():
        perm_rows.append({
            "indicator": label,
            "system":    sys_id,
            "obs_ON-OFF_diff": obs,
            "p_perm_block":   p_perm,
        })

perm_table = pd.DataFrame(perm_rows)
print()
print(f"Block-level permutation test ({N_PERMUTATIONS} permutations):")
print(perm_table.round(4).to_string(index=False))


## Section 8 — DISPROVE: regression with confounders

In [ ]:

# Section 8 — DISPROVE: regression controlling for temperature & date
# ---------------------------------------------------------------------------
# Fit y = β0 + β1·is_ON + β2·temp_mean + β3·date_index + ε  per (system, indicator).
# Report β1 and its t-stat / p, with and without temperature in the model.
# If β1 has the right sign before adding temperature but flips or shrinks to
# zero afterward, temperature was the real driver.

post = contrast[contrast["condition"].isin(["ON", "OFF"])].copy()
post["is_ON"] = (post["condition"] == "ON").astype(int)


def fit_ols(y, X):
    """Tiny OLS so we don\'t pull in statsmodels.  Returns (beta, se, t, p)
    for the FIRST regressor in X (after intercept). X should NOT include
    intercept; we add it. Two-sided t-test for that coefficient."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = ~np.isnan(y) & ~np.isnan(X).any(axis=1)
    X, y = X[keep], y[keep]
    if len(y) < X.shape[1] + 2:
        return (np.nan, np.nan, np.nan, np.nan, len(y))
    X_int = np.column_stack([np.ones(len(y)), X])
    coef, _, _, _ = np.linalg.lstsq(X_int, y, rcond=None)
    yhat = X_int @ coef
    resid = y - yhat
    dof = len(y) - X_int.shape[1]
    sigma2 = float(np.sum(resid ** 2) / dof) if dof > 0 else np.nan
    XtX_inv = np.linalg.pinv(X_int.T @ X_int)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))
    beta_on = float(coef[1])
    se_on   = float(se[1])
    t = beta_on / se_on if se_on > 0 else np.nan
    p = 2 * (1 - stats.t.cdf(abs(t), df=dof)) if not np.isnan(t) else np.nan
    return (beta_on, se_on, t, p, len(y))


reg_rows = []
for col, label in INDICATORS:
    for sys_id in SYSTEMS:
        sub = post[post["system_id"] == sys_id].copy()
        # Model A: condition only
        beta_a, se_a, t_a, p_a, n_a = fit_ols(sub[col], sub[["is_ON"]])
        # Model B: condition + temperature
        if sub["temp_mean"].notna().sum() > 5:
            beta_b, se_b, t_b, p_b, n_b = fit_ols(
                sub[col], sub[["is_ON","temp_mean"]])
        else:
            beta_b, se_b, t_b, p_b, n_b = (np.nan,)*5
        # Model C: condition + temperature + date_index
        if sub["temp_mean"].notna().sum() > 5:
            beta_c, se_c, t_c, p_c, n_c = fit_ols(
                sub[col], sub[["is_ON","temp_mean","date_index"]])
        else:
            beta_c, se_c, t_c, p_c, n_c = (np.nan,)*5

        reg_rows.append({
            "indicator": label,
            "system":    sys_id,
            "n":         int(n_a),
            "β_ON_alone":       round(beta_a, 3),
            "p_alone":          round(p_a,    4),
            "β_ON_after_T":     round(beta_b, 3),
            "p_after_T":        round(p_b,    4),
            "β_ON_after_T_date": round(beta_c, 3),
            "p_after_T_date":    round(p_c,    4),
        })

reg = pd.DataFrame(reg_rows)
print("Linear regression — coefficient of `is_ON` before and after adding controls:")
print(reg.to_string(index=False))
print()
print("Interpretation: if β_ON_alone is large/significant but shrinks toward")
print("zero / loses significance after adding temperature & date, the apparent")
print("5G effect was being driven by those confounders.")


## Section 9 — BASELINE robustness (with / without 2026-04-18)

In [ ]:

# Section 9 — BASELINE robustness check (with/without 2026-04-18)
# ---------------------------------------------------------------------------
# 2026-04-18 has unusually high counts on system 900 (~281 exits, 277 returns).
# If the BASELINE summary is driven by that one day, "BASELINE is higher than
# experiment" is misleading.

base = daily[daily["condition"] == "BASELINE"].copy()
print(f"BASELINE days: {sorted(base['date'].unique().tolist())}")

for sys_id in SYSTEMS:
    sub = base[base["system_id"] == sys_id]
    if sub.empty:
        continue
    with_18    = sub
    without_18 = sub[sub["date"] != "2026-04-18"]
    print(f"\nSystem {sys_id} BASELINE:")
    print(f"  with 2026-04-18:    n={len(with_18):2d}  mean exits = {with_18['n_exit_v3'].mean():.1f}  "
          f"mean returns = {with_18['n_return_v3'].mean():.1f}")
    print(f"  without 2026-04-18: n={len(without_18):2d}  mean exits = {without_18['n_exit_v3'].mean():.1f}  "
          f"mean returns = {without_18['n_return_v3'].mean():.1f}")

# Compare BASELINE-without-04-18 to experiment OFF days
print()
print("Cleaned BASELINE vs experiment OFF (mean ± std exit+return / day):")
for sys_id in SYSTEMS:
    sub = daily[daily["system_id"] == sys_id]
    b = sub[(sub["condition"] == "BASELINE") & (sub["date"] != "2026-04-18")]
    off = sub[sub["condition"] == "OFF"]
    if b.empty or off.empty:
        continue
    bvals = b["exit_return_count"]
    ovals = off["exit_return_count"]
    print(f"  system {sys_id}: BASELINE = {bvals.mean():.1f} ± {bvals.std():.1f}  (n={len(bvals)})    "
          f"OFF = {ovals.mean():.1f} ± {ovals.std():.1f}  (n={len(ovals)})")


## Section 10 — Verdict table

In [ ]:

# Section 10 — Verdict table
# ---------------------------------------------------------------------------

def lookup_p(table, system, indicator, col):
    sub = table[(table["system"] == system) & (table["indicator"] == indicator)]
    if sub.empty:
        return np.nan
    return float(sub[col].values[0])


claims = []

# A) "5G hinders bumblebees in general" — would need BOTH systems
for col, label in INDICATORS:
    p900 = lookup_p(effect_table, 900, label, "p_BH")
    p939 = lookup_p(effect_table, 939, label, "p_BH")
    r900 = lookup_p(effect_table, 900, label, "r")
    r939 = lookup_p(effect_table, 939, label, "r")
    same_sign = (not pd.isna(r900) and not pd.isna(r939)
                 and np.sign(r900) == np.sign(r939))
    claims.append({
        "claim": f"H1 (general): 5G changes {label} on BOTH systems",
        "support":     "yes" if (same_sign and abs(r900) > 0.3 and abs(r939) > 0.3) else "no",
        "p_BH_900":    p900,
        "p_BH_939":    p939,
        "note":        "needs same direction + |r|>0.3 on BOTH",
    })

# B) "5G hinders bumblebees on system 939 (dose-response with distance)"
for col, label in INDICATORS:
    r939 = lookup_p(effect_table, 939, label, "r")
    p_uncorrected = lookup_p(effect_table, 939, label, "p")
    p_bh           = lookup_p(effect_table, 939, label, "p_BH")
    perm_row = perm_table[(perm_table["system"] == 939)
                          & (perm_table["indicator"] == label)] if "perm_table" in dir() else pd.DataFrame()
    p_perm = float(perm_row["p_perm_block"].values[0]) if not perm_row.empty else np.nan
    claims.append({
        "claim": f"H1 (sys 939): ON changes {label} on the closer system",
        "support":     ("strong" if (abs(r939) > 0.3 and p_bh < 0.05)
                        else "moderate" if (abs(r939) > 0.3 and p_perm < 0.10 and not pd.isna(p_perm))
                        else "weak" if (abs(r939) > 0.3)
                        else "no"),
        "r_939":           r939,
        "p_uncorr_939":    p_uncorrected,
        "p_BH_939":        p_bh,
        "p_perm_block_939": p_perm,
        "note":             "block-permutation p is the most defensible",
    })

verdict = pd.DataFrame(claims)
pd.set_option("display.max_colwidth", None)
print(verdict.round(3).to_string(index=False))


## Section R.1 — Bootstrap 95% CIs on effect sizes

In [ ]:

# Section R — Academic-rigor checks
# ---------------------------------------------------------------------------
# Five things that turn a "preliminary finding" into a defensible one:
#   R.1  Bootstrap 95% CIs on the rank-biserial effect sizes.
#   R.2  Three-way comparison BASELINE / OFF / ON  (treatment validity).
#   R.3  Cohen\'s benchmarks to calibrate "how big is the effect, really".
#   R.4  Family-wise BH across both systems (8 tests, not 4).
#   R.5  Threshold-proximity sensitivity — how many tracks are within ±10%
#        of a v3 decision boundary?  If few, the headline is robust.
# ---------------------------------------------------------------------------

# ===========================================================================
# R.1 Bootstrap 95% CIs on effect sizes
# ===========================================================================

def bootstrap_r_ci(on_vals, off_vals, n_boot=5000, ci=0.95, seed=20260516):
    rng = np.random.default_rng(seed)
    on_v, off_v = np.asarray(on_vals), np.asarray(off_vals)
    if len(on_v) < 2 or len(off_v) < 2:
        return (np.nan, np.nan)
    rs = np.empty(n_boot)
    for i in range(n_boot):
        b_on  = rng.choice(on_v,  size=len(on_v),  replace=True)
        b_off = rng.choice(off_v, size=len(off_v), replace=True)
        rs[i] = rank_biserial(b_on, b_off)
    alpha = (1 - ci) / 2
    return float(np.percentile(rs, 100 * alpha)), float(np.percentile(rs, 100 * (1 - alpha)))


rows = []
for sys_id in SYSTEMS:
    sub = contrast[(contrast["system_id"] == sys_id)
                   & contrast["condition"].isin(["ON", "OFF"])]
    for col, label in INDICATORS:
        on  = sub.loc[sub["condition"] == "ON",  col].dropna()
        off = sub.loc[sub["condition"] == "OFF", col].dropna()
        r = rank_biserial(on, off) if (len(on) and len(off)) else np.nan
        lo, hi = bootstrap_r_ci(on, off, n_boot=2000)
        crosses_zero = (not pd.isna(lo)) and (lo < 0 < hi)
        rows.append({
            "system": sys_id, "indicator": label, "n_ON": len(on), "n_OFF": len(off),
            "r":   round(r, 3) if not pd.isna(r) else np.nan,
            "CI_low":  round(lo, 3) if not pd.isna(lo) else np.nan,
            "CI_high": round(hi, 3) if not pd.isna(hi) else np.nan,
            "CI_crosses_0": crosses_zero,
        })
ci_table = pd.DataFrame(rows)
print("R.1 — Bootstrap 95% CIs on rank-biserial r (ON vs OFF):")
print(ci_table.to_string(index=False))
print()
print("If a CI crosses zero, the direction of the effect is not even")
print("well-determined, regardless of the point estimate.")


## Section R.2 — Three-way BASELINE / OFF / ON comparison

In [ ]:
# ===========================================================================
# R.2 Three-way BASELINE / OFF / ON comparison
# ===========================================================================
# Two questions:
#   (a) Are OFF days different from BASELINE? If so, the equipment being
#       present at all has an effect — not just the radiation.
#   (b) Are ON days different from BASELINE? If yes AND OFF ≈ BASELINE,
#       that\'s the cleanest possible support for an ON-specific effect.

rows = []
for sys_id in SYSTEMS:
    sub = contrast[contrast["system_id"] == sys_id]
    for col, label in INDICATORS:
        base = sub.loc[sub["condition"] == "BASELINE", col].dropna()
        off  = sub.loc[sub["condition"] == "OFF",       col].dropna()
        on   = sub.loc[sub["condition"] == "ON",        col].dropna()

        # Kruskal-Wallis omnibus test
        groups = [g for g in (base, off, on) if len(g) >= 2]
        kw_p = stats.kruskal(*groups).pvalue if len(groups) >= 2 else np.nan

        # Pairwise MWU
        def _mwu(a, b):
            if len(a) < 2 or len(b) < 2:
                return np.nan
            return float(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)

        rows.append({
            "system": sys_id, "indicator": label,
            "n_BASELINE": len(base), "n_OFF": len(off), "n_ON": len(on),
            "med_BASELINE": round(float(base.median()), 2) if len(base) else np.nan,
            "med_OFF":      round(float(off.median()),  2) if len(off)  else np.nan,
            "med_ON":       round(float(on.median()),   2) if len(on)   else np.nan,
            "p_KW_omnibus":   round(kw_p, 4) if not pd.isna(kw_p) else np.nan,
            "p_BASE_vs_OFF":  round(_mwu(base, off), 4),
            "p_BASE_vs_ON":   round(_mwu(base, on),  4),
            "p_OFF_vs_ON":    round(_mwu(off,  on),  4),
        })

baseline_table = pd.DataFrame(rows)
print("R.2 — Three-way BASELINE / OFF / ON comparison:")
print(baseline_table.to_string(index=False))
print()
print("Reading: if p_BASE_vs_OFF is large (≥0.10) AND p_BASE_vs_ON is small,")
print("the equipment isn\'t the problem and ON is the problem.")
print("If p_BASE_vs_OFF is small (<0.10), the equipment alone causes a shift —")
print("any ON effect cannot be cleanly attributed to radiation.")


## Section R.3 — Effect-size magnitudes (Cohen's benchmarks)

In [ ]:
# ===========================================================================
# R.3 Effect-size interpretation against Cohen\'s benchmarks
# ===========================================================================
# Cohen (1988) for r:  small ≥ 0.10,  medium ≥ 0.30,  large ≥ 0.50.
# These are convention, not law — but readers expect them.

def cohen_label(r):
    if pd.isna(r):
        return "—"
    ar = abs(r)
    if ar >= 0.50: return "LARGE"
    if ar >= 0.30: return "medium"
    if ar >= 0.10: return "small"
    return "negligible"


magnitude = ci_table.copy()
magnitude["magnitude"] = magnitude["r"].apply(cohen_label)
magnitude["sign"]      = magnitude["r"].apply(
    lambda x: "—" if pd.isna(x) else ("+" if x > 0 else "−"))

print("R.3 — Effect-size magnitudes (Cohen\'s benchmarks on |r|):")
print(magnitude[["system","indicator","r","CI_low","CI_high",
                  "magnitude","sign","CI_crosses_0"]].to_string(index=False))
print()
print("A medium effect that fails to clear zero in its 95% CI is")
print("\"suggestive but not established\".  A large effect with a CI safely")
print("away from zero is the only category that supports a confident claim.")


## Section R.4 — Family-wise BH correction across both systems

In [ ]:
# ===========================================================================
# R.4 Family-wise BH correction — across BOTH systems (8 tests)
# ===========================================================================
# The exposure_analysis Section K does BH per-system (4 tests).  That\'s the
# right unit if you\'ve pre-specified per-system analysis.  If you\'re
# treating "any indicator × any system" as one family of tests (more
# common framing), BH should run across 4 × 2 = 8 p-values.  This is more
# conservative — and arguably more appropriate when the headline claim is
# "5G hinders bumblebees" (not yet pinned to one system).

from scipy.stats import false_discovery_control

pvals_all = []
labels_all = []
for sys_id in SYSTEMS:
    sub = contrast[(contrast["system_id"] == sys_id)
                   & contrast["condition"].isin(["ON", "OFF"])]
    for col, label in INDICATORS:
        on  = sub.loc[sub["condition"] == "ON",  col].dropna()
        off = sub.loc[sub["condition"] == "OFF", col].dropna()
        if len(on) >= 2 and len(off) >= 2:
            _, p = stats.mannwhitneyu(on, off, alternative="two-sided")
        else:
            p = np.nan
        pvals_all.append(p)
        labels_all.append((sys_id, label))

# BH on the 8 valid p-values
valid_idx = [i for i, p in enumerate(pvals_all) if not pd.isna(p)]
valid_p   = [pvals_all[i] for i in valid_idx]
adj       = false_discovery_control(valid_p) if valid_p else []
adj_full  = [np.nan] * len(pvals_all)
for i, a in zip(valid_idx, adj):
    adj_full[i] = float(a)

family_table = pd.DataFrame({
    "system":    [s for s, _ in labels_all],
    "indicator": [l for _, l in labels_all],
    "p_raw":     [round(p, 4) if not pd.isna(p) else np.nan for p in pvals_all],
    "p_BH_per_system":  None,
    "p_BH_family_wise": [round(a, 4) if not pd.isna(a) else np.nan for a in adj_full],
    "survives_q05_familywise": [a < 0.05 if not pd.isna(a) else False for a in adj_full],
})

# Per-system BH for comparison
for sys_id in SYSTEMS:
    mask = family_table["system"] == sys_id
    p_sys = family_table.loc[mask, "p_raw"].tolist()
    valid_sys_idx = [i for i, p in enumerate(p_sys) if not pd.isna(p)]
    if valid_sys_idx:
        adj_sys = false_discovery_control([p_sys[i] for i in valid_sys_idx])
        out = [np.nan] * len(p_sys)
        for i, a in zip(valid_sys_idx, adj_sys):
            out[i] = float(a)
        family_table.loc[mask, "p_BH_per_system"] = [
            round(a, 4) if not pd.isna(a) else np.nan for a in out
        ]

print("R.4 — BH correction: per-system (4 tests) vs family-wise (8 tests):")
print(family_table.to_string(index=False))
print()
n_per  = int(family_table["p_BH_per_system"].lt(0.05).sum())
n_fam  = int(family_table["survives_q05_familywise"].sum())
print(f"Survives BH per-system at q=0.05:  {n_per} / 8")
print(f"Survives BH family-wise at q=0.05: {n_fam} / 8")
print()
print("The family-wise number is the one to report if your headline claim")
print("isn\'t pre-restricted to a single system.  If it\'s zero, the honest")
print("write-up is \"no indicator survives correction\" plus the directional")
print("flag from Section K as suggestive evidence requiring replication.")


## Section R.5 — Threshold-proximity sensitivity

In [ ]:
# ===========================================================================
# R.5 Threshold-proximity sensitivity check
# ===========================================================================
# A genuinely robust headline shouldn\'t hinge on a few borderline tracks.
# For each v3 decision threshold, count how many per-track feature values
# sit within ±10% of the boundary. High proximity (>20%) = sensitive
# headline; low proximity (<5%) = robust headline.

THRESHOLDS = [
    ("initial_speed_min",  0.5,  "speed < 0.5 (takeoff)"),
    ("terminal_speed_min", 0.5,  "speed < 0.5 (landing)"),
    ("start_dist",         0.30, "start ≤ 0.30 m"),
    ("end_dist",           0.20, "end ≤ 0.20 m"),
    ("outward_cos",        0.30, "outward cos > 0.30"),
    ("toward_cos",         0.30, "toward cos > 0.30"),
]

rows = []
for col, thr, label in THRESHOLDS:
    if col not in tracks.columns:
        continue
    for sys_id in SYSTEMS:
        sub = tracks[(tracks["system_id"] == sys_id) & tracks[col].notna()]
        if sub.empty:
            continue
        delta = 0.1 * abs(thr) if thr != 0 else 0.05
        in_band = sub[(sub[col] >= thr - delta) & (sub[col] <= thr + delta)]
        rows.append({
            "system":  sys_id,
            "feature": label,
            "n_total":     len(sub),
            "n_in_±10%":   len(in_band),
            "frac_in_band": round(len(in_band) / len(sub), 3),
        })

sens_table = pd.DataFrame(rows)
print("R.5 — Threshold-proximity sensitivity:")
print(sens_table.to_string(index=False))
print()
print("Reading: frac_in_band > 0.20 means >20% of tracks are within ±10% of")
print("the threshold — those classifications would flip with a small parameter")
print("tweak. The headline depends on a sensitive boundary in that case.")
print("frac < 0.05 ⇒ the threshold sits in a thin part of the distribution")
print("(robust). Typically `end_dist` and `start_dist` are the most sensitive.")


# ===========================================================================
# R.5b — Export borderline tracks for 3D-viewer inspection
# ===========================================================================
# For each (feature, system), grab the 20 UIDs closest to the threshold.
# Load these into the PATS-C 3D viewer and confirm by eye whether they
# look like fly-bys (should be rejected) or real exits/returns (should be
# accepted). This is the visual-validation step for R.5.

OUT_BORDERLINE = V3_DIR / "borderline_tracks_for_inspection.csv"
N_PER_BUCKET = 20

borderline_rows = []
for col, thr, label in THRESHOLDS:
    if col not in tracks.columns:
        continue
    relevant_classification = ("hive_exit_v3"
                                if col in ("initial_speed_min", "start_dist", "outward_cos")
                                else "hive_return_v3")
    for sys_id in SYSTEMS:
        sub = tracks[(tracks["system_id"] == sys_id) & tracks[col].notna()].copy()
        if sub.empty:
            continue
        sub["dist_to_thr"] = (sub[col] - thr).abs()
        nearest = sub.nsmallest(N_PER_BUCKET, "dist_to_thr")
        for _, row in nearest.iterrows():
            borderline_rows.append({
                "feature":         col,
                "feature_label":   label,
                "threshold":       thr,
                "system_id":       sys_id,
                "uid":             row["uid"],
                "date":            row["date"],
                "feature_value":   round(float(row[col]), 4),
                "dist_to_thr":     round(float(row["dist_to_thr"]), 4),
                "v3_classification": bool(row[relevant_classification]),
                "n_samples":       int(row["n_samples"]),
            })

borderline = pd.DataFrame(borderline_rows)
borderline.to_csv(OUT_BORDERLINE, index=False)
print()
print(f"Wrote {len(borderline)} borderline UIDs → {OUT_BORDERLINE}")
print()
print("How to use this list:")
print("  1. Load each UID into the PATS-C 3D viewer.")
print("  2. For each track, eyeball whether the trajectory looks like:")
print("       (a) a fly-by (passes near hive but doesn\'t land) → v3 SHOULD reject")
print("       (b) a real exit/return that v3 might miss → v3 SHOULD accept")
print("  3. If the v3_classification column matches your visual call, the")
print("     threshold is well-placed. If it disagrees on most borderline tracks,")
print("     the threshold needs nudging.")
print()
print("Suggested minimum: 5 tracks per (feature, system) — picks at the extreme")
print("ends of the band. Focus on `end_dist` and `start_dist` first since those")
print("are the ones the data show as most sensitive.")
print()
print(borderline.groupby(["feature_label","system_id","v3_classification"])
                  .size().unstack(fill_value=0))


## How to read the verdict table

- **Claim A (general H1 — hinders bumblebees broadly):** would require BOTH
  systems to show the effect in the same direction with at least medium effect
  size on each. If the table shows "no" here, the broad claim is not supported.
- **Claim B (system-939 H1 — closer to source):** the more defensible framing.
  Look at `p_perm_block_939` rather than the raw or BH-corrected p — it's the
  test that correctly accounts for 3-day cycle non-independence. Anything ≤ 0.10
  is suggestive, ≤ 0.05 is moderate; nothing in this dataset will hit ≤ 0.01.

## Recommended framing for the professor

The data are *consistent with* a hypothesis that 5G exposure on the system
closer to the antenna depresses foraging activity and the return ratio, with
elevated path tortuosity as a corroborating finding. They do **not** support
a generalized "5G hinders bumblebees" claim because the more-distant system
(900) shows no effect.

Caveats that should accompany any claim:

1. Sample size is small (n ≈ 9-11 days per condition per system) and days
   within 3-day blocks aren't independent.
2. Temperature is correlated with activity (Section 2); the apparent ON
   effect on count partially survives temperature adjustment (Section 8).
3. Treatment verification (Section 1) shows dBm did/did-not (whichever it is
   when you run this) differ between ON and OFF — that's the precondition.
4. No replication. A single experiment in one greenhouse with two hives.

What would strengthen the claim: more cycles (8+ instead of 3-4), a second
greenhouse, sham-treatment days (turn the equipment on but transmit nothing)
to rule out cooling-fan / EMI artifacts, and per-bee tracking to see whether
the same individuals are affected differently across cycles.
